### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
# Read all pdf's file

def process_all_pdfs(pdf_directory):
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"    Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [18]:
all_pdf_docs = process_all_pdfs("../data/")

Found 5 PDF files to process.

 Processing: AI Learning Resources (Beyond the Basics).pdf
    Loaded 2 pages

 Processing: AI_ML_Master_Roadmap_2026_Outline.pdf
    Loaded 2 pages

 Processing: 3_Month_AI_ML_Roadmap.pdf
    Loaded 1 pages

 Processing: Design_Patterns_Interview_Revision.pdf
    Loaded 1 pages

 Processing: Week 1.pdf
    Loaded 57 pages

Total documents loaded: 63


In [22]:
# Text Splitting - Chunking

def split_documents(documents, chunk_size=500, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [23]:
chunks = split_documents(all_pdf_docs)

Split 63 documents into 72 chunks

Example chunk: 
Content: 🚀  AI  Learning  Resources  (Beyond  the  
Basics)
 
If  you're  serious  about  AI,  here  are  some  of  the  resources  that  will  make  you  stand  out.  
🧠  Learn  how  great  engineers  think  ...
Metadata: {'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': '🚀 AI Learning Resources (Beyond the Basics)', 'source': '../data/pdf/AI Learning Resources (Beyond the Basics).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'AI Learning Resources (Beyond the Basics).pdf', 'file_type': 'pdf'}


#### Embedding and Vector DB

In [29]:
import uuid
import chromadb
import warnings
import numpy as np
warnings.filterwarnings('ignore')
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class EmbeddingManager:
    
    def __init__(self, model_name: str="all-MiniLM-L6-v2"): 
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully. Embedded Dimension: {self.model.get_embedding_dimension()}")
            
        except Exception as e:
            print(f"Error loading model {e}")
            raise
        
    def generate_embedding(self, text: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embedding for {len(text)} texts")
        embedding = self.model.encode(text, show_progress_bar=True)
        print(f"Generating embedding with shape: {len(embedding.shape)}")
        
        return embedding 

In [31]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3903.86it/s]


Model loaded Successfully. Embedded Dimension: 384


#### Vector Store

In [38]:
class VectorStore:
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        
        try: 
            # Create Persistant client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or Create Collections
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata= {
                    "Description": "PDF documents embedding for RAG"
                }
            )
            
            print(f"Vector Store Initialized. Collection: {self.collection_name}")
            print(f"Existing Documents in collections: {self.collection.count()}")
        
        except Exception as e:
            
            print(f"Error Initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        if len(documents) != len(embeddings):
            raise ValueError("No of Documents must match the number of embeddings")

        print(f"Adding {len(documents)} documents to Vector Store")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            
            # Generating IDs
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Generating Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Documents Content
            documents_text.append(doc.page_content)
            
            # Embeddings
            embeddings_list.append(embedding.tolist())
            
        try: 
            self.collection.add(
                ids = ids,
                metadatas= metadatas,
                embeddings= embeddings_list,
                documents= documents_text
            )
            
            print(f"Added {len(documents)} documents to Vector Store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to Vector Store {e}")
            raise

In [39]:
vector_store = VectorStore()
vector_store

Vector Store Initialized. Collection: pdf_documents
Existing Documents in collections: 72


In [40]:
# Converting text to embeddings
texts = [doc.page_content for doc in chunks]

# Generating Embeddings
embedding = embedding_manager.generate_embedding(texts)

# Storing in Vector Database
vector_store.add_documents(chunks, embedding)

Generating embedding for 72 texts


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.46it/s]

Generating embedding with shape: 2
Adding 72 documents to Vector Store
Added 72 documents to Vector Store
Total documents in collection: 144
